<a href="https://colab.research.google.com/github/parkhaG1t/encryption/blob/main/assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random
import time

# 서버가 시작된 시간(Seed)을 알면 난수를 예측할 수 있다!
# 가정: 해커가 알아낸 서버 시작 시간(Seed)은 20250101 입니다.
server_seed = 20250314
random.seed(server_seed)

# 서버는 이미 5번 주사위를 굴렸습니다.
for _ in range(5):
    random.randint(1, 6)

In [2]:
next_values = [random.randint(1, 6) for _ in range(3)]

print(f"예측된 주사위 값: {next_values}")

예측된 주사위 값: [2, 6, 3]


In [4]:
next_values = [random.randint(1, 6) for _ in range(3)]

print(f"실제 주사위 값: {next_values}")

실제 주사위 값: [2, 6, 3]


In [5]:
def xor_hex(hex1, hex2):
  # 두 16진수 문자열을 XOR 하여 16진수 문자열로 반환하는 함수 작성
  # 1. hex string -> int 변환 (int(h, 16))
  # 2. XOR 연산
  # 3. 결과 -> hex string 변환
  int1 = int(hex1, 16)
  int2 = int(hex2, 16)

  xor_result = int1 ^ int2

  return hex(xor_result)[2:].zfill(len(hex1))

c1 = "3278635d1149276b542c"
c2 = "29786c431b5c286d532c"

# P1 추측: "HELLOALICE" (아스키 코드를 Hex로 변환하여 사용)
p1_str = "HELLOALICE"
p1_hex = "".join([hex(ord(c))[2:].zfill(2) for c in p1_str])

# P2 = C1 ^ C2 ^ P1
c1_xor_c2 = xor_hex(c1, c2) # 문제1
p2_hex = xor_hex(c1_xor_c2, p1_hex)

p2_res = ""
for i in range(0, len(p2_hex), 2):
  p2_res += chr(int(p2_hex[i:i+2], 16))

print(f"복구된 p2 : {p2_res}")  # 문제2

# 문제 3 : SECRETCODE

복구된 p2 : SECRETCODE


In [7]:
cipher_hex = "091600175a0c015856410217090058100700160007"
cipher = bytearray.fromhex(cipher_hex)

old_text = "user"
new_text = "root"

start_index = len(cipher) -4

for i in range(4) :
  mask = ord(old_text[i]) ^ ord(new_text[i])  # mask에 user 와 root xor 후 저장
  cipher[start_index + i] ^= mask             # 암호문에 user 부분을 root를 암호화한 값 저장

print(f"변조된 암호문(Hex): {cipher.hex()}")

# 복호화
original_plaintext = "user_id=100&role=user"
original_plaintext_bytes = original_plaintext.encode()

original_cipher_hex = "091600175a0c015856410217090058100700160007"
original_cipher_bytes = bytes.fromhex(original_cipher_hex)

keystream = bytes([p ^ c for p, c in zip(original_plaintext_bytes, original_cipher_bytes)])

decode_bytes = bytes([c ^ k for c, k in zip(cipher, keystream)])

print(f"복호화된 평문: {decode_bytes.decode()}")

변조된 암호문(Hex): 091600175a0c0158564102170900581007070a0a01
복호화된 평문: user_id=100&role=root


In [10]:
class RC4:
    def __init__(self, key):
        self.S = list(range(256))
        # KSA 구현
        if isinstance(key, str):
          key = [ord(c) for c in key]   # key를 바이트로 변환

        j = 0
        for i in range(256):
          j = (j + self.S[i] + key[i % len(key)]) % 256   # j 인덱스 값을 입력받은 key를 통해서 섞기

          self.S[i], self.S[j] = self.S[j], self.S[i]     # swap

    def encrypt(self, text):
        # PRGA 및 XOR 구현
        # 입력받은 평문 바이트로 변환
        if isinstance(text, str):
          data = [ord(c) for c in text]
        else:
          data = text

        S = self.S[:]   # 원본 S-box 저장

        res = []        # 암호화된 결과 값 저장할 리스트
        i = 0
        j = 0

        for byte in data:
          i = (i+1) % 256       # 한 칸씩 이동
          j = (j+S[i]) % 256    # S-box 값만큼 이동

          S[i], S[j] = S[j], S[i]   # swap -> 계속 S-box 값이 섞임

          K = S[(S[i] + S[j]) % 256]    # 랜덤한 값 추출

          res.append(byte ^ K)

        return bytes(res)


# 테스트
key = "Secret"
plaintext = "Attack at dawn"

rc4_cipher = RC4(key)

encrypted = rc4_cipher.encrypt(plaintext)

print(f"Key: {key}")
print(f"Plaintext: {plaintext}")
print(f"암호문(Hex): {encrypted.hex()}") # 16진수 문자열 출력

decrypted = rc4_cipher.encrypt(encrypted)
print(f"복호화 결과 : {decrypted.decode()}")


Key: Secret
Plaintext: Attack at dawn
암호문(Hex): 45a01f645fc35b383552544b9bf5
복호화 결과 : Attack at dawn
